# Phase 3 — BrainSegFounder Single-Modality (T1c) Fine-Tuning (v3)

**v2 fixes applied:**
1. **Target: ET** (label==2) instead of WT — T1c can actually see enhancing tumor
2. **Gradient accumulation ×4** — stabilizes noisy Dice gradients (eff batch=8)
3. **CosineAnnealingWarmRestarts** — prevents LR death at epoch 44
4. **No freeze** — CNN is compact (16.7M), all params train from epoch 0

In [1]:
MODEL_NAME = "MetSeg-CNN"
# ╔════════════════════════════════════════════════════════════╗
# ║  MetSeg T1c → ET — FOLD-BY-FOLD (v2 retrain)            ║
# ╚════════════════════════════════════════════════════════════╝

CONFIG = {
    'seg_params': {
        'spatial_dims': 3,
        'in_channels':  1,   # ← T1c ONLY
        'out_channels': 1,   # ← binary ET (enhancing tumor)
        'kernel_size':  [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
        'strides':      [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'upsample_kernel_size': [[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
        'deep_supervision': True, 'deep_supr_num': 3,
        'filters': [32, 64, 128, 256, 320], 'res_block': True, 'trans_bias': True,
    },
    'patch_size':     [64, 64, 64],
    'num_samples':    3,
    'pos_neg_ratio':  [2, 1],
    # ── Optimizer (v2) ──
    'lr':             1e-4,
    'min_lr':         1e-6,
    'warmup_epochs':  3,
    'restart_period': 25,       # CosineWarmRestarts T_0
    'restart_mult':   2,        # T_mult
    'weight_decay':   1e-5,
    'grad_accum':     4,        # eff batch=4×4=16        # effective batch = batch_size × 4
    # ─────────────────────────────────────────────────────────
    'cache_rate':     0.5,
    'batch_size':     4,
    'num_workers':    0,
    'epochs':         100,
    'patience':       25,
    'val_interval':   1,
}

ROI_SIZE    = (64, 64, 64)
ROI_PADDING = 8

print("CONFIG v2 — 4 FIXES APPLIED")
print(f"  Target  = ENHANCING TUMOR (label==2) — T1c can see this!")
print(f"  GradAcc = {CONFIG['grad_accum']}  (eff batch={CONFIG['batch_size']*CONFIG['grad_accum']})")
print(f"  LR      = warmup {CONFIG['warmup_epochs']}ep → CosineWarmRestarts T₀={CONFIG['restart_period']} T_mult={CONFIG['restart_mult']}")
print(f"  Freeze  = NONE (CNN is compact)")

CONFIG v2 — 4 FIXES APPLIED
  Target  = ENHANCING TUMOR (label==2) — T1c can see this!
  GradAcc = 4  (eff batch=16)
  LR      = warmup 3ep → CosineWarmRestarts T₀=25 T_mult=2
  Freeze  = NONE (CNN is compact)


In [2]:
import subprocess, sys
for pkg in ['monai', 'nibabel']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies ✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 77.3 MB/s eta 0:00:00
Dependencies ✅


In [3]:
import warnings; warnings.filterwarnings('ignore')
import os, json, time, shutil
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from tqdm import tqdm

from monai.networks.nets import DynUNet
from monai.losses import DiceLoss, DiceFocalLoss
from monai.data import DataLoader, CacheDataset
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
import monai.transforms as T
from monai.utils import ensure_tuple_rep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

2026-04-19 13:45:07.606480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776606307.809712      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776606307.868971      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776606308.336628      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776606308.336672      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776606308.336679      23 computation_placer.cc:177] computation placer alr

Device: cuda
GPU:  Tesla T4
VRAM: 14.6 GB


In [4]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_ROOT = Path('/kaggle/input/datasets/boufafamoamed/cyprus-proteas-brain-mets')
if not DATA_ROOT.exists():
    for c in Path('/kaggle/input').iterdir():
        if c.is_dir() and any((c/f'P{i:02d}').exists() for i in range(1,5)):
            DATA_ROOT = c; break
        if c.is_dir():
            for sub in c.iterdir():
                if sub.is_dir() and (sub/'data_splits.json').exists():
                    DATA_ROOT = sub; break

OUTPUT_ROOT = Path('/kaggle/working/phase2_metseg_t1c_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def resolve_path(root, rel):
    p = root / rel
    if p.exists(): return str(p)
    gz = str(p) + '.gz'
    if Path(gz).exists(): return gz
    nii_gz = str(p).replace('.nii.gz', '.nii_gz')
    if Path(nii_gz).exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(nii_gz, str(link))
        return str(link)
    if p.parent.exists():
        for f in p.parent.iterdir():
            if f.name.lower() == p.name.lower(): return str(f)
    raise FileNotFoundError(f'Not found: {rel}')

# ── Load splits ─────────────────────────────────────────────────────────────
def find_splits():
    candidates = []
    for name in ['data_splits.json', 'cv_splits_3fold.json', 'cv_splits.json']:
        p = DATA_ROOT / name
        if p.exists(): candidates.append(p)
    for f in DATA_ROOT.rglob('*splits*.json'):
        if f not in candidates: candidates.append(f)
    for d in Path('/kaggle/input').iterdir():
        if not d.is_dir(): continue
        p = d / 'data_splits.json'
        if p.exists() and p not in candidates: candidates.append(p)
        for sub in d.iterdir():
            if sub.is_dir():
                p = sub / 'data_splits.json'
                if p.exists() and p not in candidates: candidates.append(p)
    # Prefer 4-variable stratification
    for c in candidates:
        try:
            d = json.load(open(c))
            if len(d.get('metadata',{}).get('stratification_variables',[])) >= 4:
                return c, d
        except: pass
    for c in candidates:
        try:
            d = json.load(open(c))
            if '3fold' in d or 'fold_0' in d: return c, d
        except: pass
    raise FileNotFoundError('No valid data_splits.json')

splits_path, raw_splits = find_splits()
all_splits = raw_splits.get('3fold', raw_splits)
if 'fold_0' not in all_splits:
    all_splits = {k: v for k, v in raw_splits.items() if k.startswith('fold_')}

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'Splits    : {splits_path.name}  → folds: {list(all_splits.keys())}')

# ── Recover checkpoints from previous sessions ─────────────────────────────
ckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
emb_dir  = OUTPUT_ROOT / 'embeddings';  emb_dir.mkdir(parents=True, exist_ok=True)
fig_dir  = OUTPUT_ROOT / 'figures';     fig_dir.mkdir(parents=True, exist_ok=True)

_recovered = []
for f in sorted(Path('/kaggle/input').rglob('metseg_t1c_fold*.pth')):
    dst = ckpt_dir / f.name
    if not dst.exists():
        shutil.copy2(str(f), str(dst)); _recovered.append(f.name)
        print(f'  Recovered: {f.name}')
for pat in ['cnn_t1c_embeddings_fold*.npz', 'cnn_t1c_embeddings_fold*_meta.json']:
    for f in sorted(Path('/kaggle/input').rglob(pat)):
        dst = emb_dir / f.name
        if not dst.exists():
            shutil.copy2(str(f), str(dst)); _recovered.append(f.name)
            print(f'  Recovered: {f.name}')
if _recovered: print(f'  Total recovered: {len(_recovered)} files')
else: print('  No previous outputs found — starting fresh')

# ── Find MetSeg pretrained weights (segmentor_full_modality.ckpt) ──────────
WEIGHTS_DIR = OUTPUT_ROOT / 'pretrained_weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

seg_weights_path = None

# 1. Check Kaggle input datasets
for f in sorted(Path('/kaggle/input').rglob('*segmentor*modality*.ckpt')):
    if f.stat().st_size > 1_000_000:
        seg_weights_path = f; break

# 2. Check WEIGHTS_DIR (from previous download)
if seg_weights_path is None:
    local = WEIGHTS_DIR / 'segmentor_full_modality.ckpt'
    if local.exists() and local.stat().st_size > 1_000_000:
        seg_weights_path = local

# 3. Download from web (Met-Seg repo)
if seg_weights_path is None:
    import subprocess
    url = 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf452df0f6b/120822/-2821463328806488135/segmentor_weight%20%28full%20modality%29.ckpt'
    dst = WEIGHTS_DIR / 'segmentor_full_modality.ckpt'
    print('  Downloading MetSeg pretrained weights...')
    result = subprocess.run(['wget', '-q', '--no-check-certificate',
        '--user-agent', 'Mozilla/5.0', '-O', str(dst), url],
        capture_output=True, text=True, timeout=300)
    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:
        seg_weights_path = dst

if seg_weights_path:
    print(f'  ✅ Pretrained weights: {seg_weights_path.name}  ({seg_weights_path.stat().st_size/1e6:.0f} MB)')
else:
    print('  ⚠️ No pretrained weights found — training from scratch')
    print('  → Add "metseg-pretrained-weights" dataset with segmentor_full_modality.ckpt')

DATA_ROOT : /kaggle/input/datasets/boufafamoamed/cyprus-proteas-brain-mets
Splits    : data_splits.json  → folds: ['fold_0', 'fold_1', 'fold_2']
  No previous outputs found — starting fresh
  ✅ Pretrained weights: segmentor_full_modality.ckpt  (179 MB)


In [5]:
# ── Data dicts — T1c ONLY (single image per scan) ─────────────────────────
def build_scan_dicts(scans, label='subset'):
    dicts, skips = [], 0
    for scan in scans:
        try:
            dicts.append({
                'image': resolve_path(DATA_ROOT, scan['t1c']),   # T1c only
                'label': resolve_path(DATA_ROOT, scan['mask']),
                'patient_dir': scan['patient_dir'], 'visit': scan['visit'],
            })
        except FileNotFoundError: skips += 1
    if skips: print(f'  Skipped {skips} in {label}')
    return dicts

def get_fold_dicts(fold):
    fd = all_splits[f'fold_{fold}']
    return (build_scan_dicts(fd['train_scans'], f'fold{fold}_train'),
            build_scan_dicts(fd['test_scans'],  f'fold{fold}_val'))

def get_all_dicts():
    all_d, seen = [], set()
    for fk in all_splits:
        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:
            key = (scan['patient_dir'], scan['visit'])
            if key not in seen:
                seen.add(key)
                try:
                    all_d.append({
                        'image': resolve_path(DATA_ROOT, scan['t1c']),
                        'label': resolve_path(DATA_ROOT, scan['mask']),
                        'patient_dir': scan['patient_dir'], 'visit': scan['visit'],
                    })
                except FileNotFoundError: pass
    return all_d

train_d, val_d = get_fold_dicts(0)
print(f'Fold 0: {len(train_d)} train | {len(val_d)} val')
print(f'Total : {len(get_all_dicts())} scans')

Fold 0: 118 train | 52 val
Total : 170 scans


In [6]:
# ── Enhancing Tumor label converter ──────────────────────────────────────
# Cyprus labels: 0=bg, 1=NCR, 2=ED, 3=ET
# T1c can SEE enhancing tumor (bright on contrast) but NOT edema
# → Target = ET only (label==2) for honest T1c segmentation

class EnhancingTumorLabeld(T.MapTransform):
    '''Cyprus mask → binary ET: label==2 → 1, everything else → 0.'''
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            d[key] = (img == 2).float().unsqueeze(0)
        return d

patch = CONFIG['patch_size']

train_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    EnhancingTumorLabeld(keys=['label']),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image', 'label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=0.3),
    T.RandAffined(
        keys=['image', 'label'], prob=0.3,
        rotate_range=[0.26, 0.26, 0.26],
        scale_range=[0.1, 0.1, 0.1],
        mode=['bilinear', 'nearest'],
        padding_mode='border'
    ),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=0.3),
    T.RandAffined(
        keys=['image', 'label'], prob=0.3,
        rotate_range=[0.26, 0.26, 0.26],
        scale_range=[0.1, 0.1, 0.1],
        mode=['bilinear', 'nearest'],
        padding_mode='border'
    ),
    T.SpatialPadd(keys=['image', 'label'], spatial_size=ensure_tuple_rep(patch, 3)),
    T.RandCropByPosNegLabeld(keys=['image', 'label'], label_key='label',
        spatial_size=ensure_tuple_rep(patch, 3),
        pos=CONFIG['pos_neg_ratio'][0], neg=CONFIG['pos_neg_ratio'][1],
        num_samples=CONFIG['num_samples'], image_key='image', image_threshold=0),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

val_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    EnhancingTumorLabeld(keys=['label']),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

print('Transforms ready ✅')
print('  Target : ENHANCING TUMOR (label==2) — what T1c can actually see')
print('  Input  : T1c (1ch)')

Transforms ready ✅
  Target : ENHANCING TUMOR (label==2) — what T1c can actually see
  Input  : T1c (1ch)


In [7]:
import math
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def create_segmenter():
    return DynUNet(**CONFIG['seg_params'])


def load_pretrained_weights_adapted(model, ckpt_path):
    '''Load MetSeg pretrained weights — SELECT T1c (ch=1) and ET output (ch=2).'''
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt)

    clean_sd = OrderedDict()
    for k, v in sd.items():
        new_k = k[len('model.'):] if k.startswith('model.') else k
        clean_sd[new_k] = v

    my_sd = model.state_dict()
    adapted_sd = OrderedDict()
    adapted_keys = []

    for k, v in clean_sd.items():
        if k not in my_sd: continue
        my_shape = my_sd[k].shape
        if v.shape == my_shape:
            adapted_sd[k] = v
        elif len(v.shape) >= 2 and v.shape[1] != my_shape[1] and my_shape[1] == 1:
            # Input: select T1c channel (idx=1)
            t1c_idx = 1 if v.shape[1] >= 2 else 0
            adapted_sd[k] = v[:, t1c_idx:t1c_idx+1, ...]
            adapted_keys.append(f'{k}: {tuple(v.shape)}→{tuple(adapted_sd[k].shape)} (T1c ch={t1c_idx})')
        elif len(v.shape) >= 1 and v.shape[0] != my_shape[0] and my_shape[0] == 1:
            # Output: select ET channel (idx=2 in [WT,TC,ET])
            et_idx = 2 if v.shape[0] >= 3 else 0
            adapted_sd[k] = v[et_idx:et_idx+1, ...]
            adapted_keys.append(f'{k}: {tuple(v.shape)}→{tuple(adapted_sd[k].shape)} (ET ch={et_idx})')
        else:
            pass

    missing, unexpected = model.load_state_dict(adapted_sd, strict=False)
    print(f'  Loaded: {len(adapted_sd)}/{len(my_sd)} params')
    if adapted_keys:
        print(f'  Adapted {len(adapted_keys)} layers:')
        for ak in adapted_keys[:5]: print(f'    {ak}')
    if missing: print(f'  Missing: {len(missing)}')
    return len(adapted_sd)


# ── Verify ──
m = create_segmenter()
p = sum(x.numel() for x in m.parameters())
print(f'DynUNet  in=1(T1c)  out=1(ET)')
print(f'Params: {p:,} ({p/1e6:.1f}M)')
with torch.no_grad():
    d = torch.zeros(1, 1, 64, 64, 64)
    o = m(d)
    oo = o[0] if isinstance(o, (list, tuple)) else o
    print(f'Forward: {tuple(d.shape)} → {tuple(oo.shape)} ✅')

if seg_weights_path:
    print(f'\nAdapting weights (T1c input, ET output)...')
    load_pretrained_weights_adapted(m, seg_weights_path)
del m

DynUNet  in=1(T1c)  out=1(ET)
Params: 16,671,108 (16.7M)
Forward: (1, 1, 64, 64, 64) → (1, 4, 1, 64, 64, 64) ✅

Adapting weights (T1c input, ET output)...
  Loaded: 168/168 params
  Adapted 18 layers:
    input_block.conv1.conv.weight: (32, 4, 3, 3, 3)→(32, 1, 3, 3, 3) (T1c ch=1)
    input_block.conv3.conv.weight: (32, 4, 1, 1, 1)→(32, 1, 1, 1, 1) (T1c ch=1)
    output_block.conv.conv.weight: (3, 32, 1, 1, 1)→(1, 32, 1, 1, 1) (ET ch=2)
    output_block.conv.conv.bias: (3,)→(1,) (ET ch=2)
    deep_supervision_heads.0.conv.conv.weight: (3, 64, 1, 1, 1)→(1, 64, 1, 1, 1) (ET ch=2)


In [8]:
def train_fold(fold):
    best_path   = ckpt_dir / f'metseg_t1c_fold{fold}_best.pth'
    latest_path = ckpt_dir / f'metseg_t1c_fold{fold}_latest.pth'

    if best_path.exists() and latest_path.exists():
        c = torch.load(latest_path, map_location='cpu', weights_only=False)
        if c.get('epoch', -1) >= CONFIG['epochs'] - 1:
            bd = c.get('best_dice', 0)
            print(f'  ✅ Fold {fold} complete (Dice={bd:.4f})')
            m = create_segmenter().to(device)
            m.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)['seg_state_dict'])
            return m, bd, True

    tr_d, va_d = get_fold_dicts(fold)
    print(f'  Train: {len(tr_d)} | Val: {len(va_d)}')

    tr_ds = CacheDataset(tr_d, train_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    va_ds = CacheDataset(va_d, val_transforms,   cache_rate=1.0,                 num_workers=0)
    tr_ld = DataLoader(tr_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=CONFIG['num_workers'], pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=1,                   shuffle=False, num_workers=0)

    model = create_segmenter()
    if seg_weights_path:
        load_pretrained_weights_adapted(model, seg_weights_path)
        print('  ✅ Pretrained weights loaded (T1c input, ET output)')

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

    # CosineAnnealingWarmRestarts — prevents LR death
    scheduler = CosineAnnealingWarmRestarts(optimizer,
        T_0=CONFIG['restart_period'], T_mult=CONFIG['restart_mult'], eta_min=CONFIG['min_lr'])
    # Warmup: manually set LR for first few epochs
    warmup_done = False

    loss_fn = DiceFocalLoss(
        sigmoid=True, gamma=2.0,
        lambda_dice=0.5, lambda_focal=0.5,
        smooth_nr=0, smooth_dr=1e-5
    )
    scaler  = torch.amp.GradScaler('cuda')
    metric  = DiceMetric(include_background=True, reduction='mean')
    accum   = CONFIG['grad_accum']

    best_dice = 0.0; patience_ctr = 0; start = 0
    log = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'lr': []}

    # Resume
    if latest_path.exists():
        c = torch.load(latest_path, map_location=device, weights_only=False)
        model.load_state_dict(c['seg_state_dict'])
        start = c['epoch'] + 1
        best_dice = c.get('best_dice', 0)
        log = c.get('log', log)
        patience_ctr = c.get('patience', 0)
        print(f'  �� Resuming from epoch {start} (best={best_dice:.4f})')

    t0 = time.time()
    for epoch in range(start, CONFIG['epochs']):
        # Warmup LR
        if epoch < CONFIG['warmup_epochs']:
            lr = 1e-5 + (CONFIG['lr'] - 1e-5) * (epoch / CONFIG['warmup_epochs'])
            for pg in optimizer.param_groups: pg['lr'] = lr
        elif not warmup_done:
            warmup_done = True
            # Reset scheduler to start from current epoch
            scheduler = CosineAnnealingWarmRestarts(optimizer,
                T_0=CONFIG['restart_period'], T_mult=CONFIG['restart_mult'], eta_min=CONFIG['min_lr'])
        
        lr = optimizer.param_groups[0]['lr']

        model.train(); ep_loss = 0; steps = 0
        optimizer.zero_grad()

        for i, batch in enumerate(tr_ld):
            imgs = batch['image'].to(device)
            lbls = batch['label'].to(device)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
                if isinstance(out, (list, tuple)):
                    loss = sum(0.5**k * loss_fn(p, lbls) for k, p in enumerate(out))
                elif out.dim() == 6:
                    loss = sum(0.5**k * loss_fn(p, lbls) for k, p in enumerate(torch.unbind(out, 1)))
                else:
                    loss = loss_fn(out, lbls)
                loss = loss / accum   # scale for accumulation

            scaler.scale(loss).backward()

            if (i + 1) % accum == 0 or (i + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            ep_loss += loss.item() * accum; steps += 1

        # Step scheduler after warmup
        if warmup_done:
            scheduler.step(epoch - CONFIG['warmup_epochs'])

        avg = ep_loss / max(steps, 1)
        log['train_loss'].append(avg); log['lr'].append(lr)

        # Validation
        if (epoch + 1) % CONFIG['val_interval'] == 0 or epoch == CONFIG['epochs'] - 1:
            model.eval(); metric.reset()
            ep_vloss = 0; vsteps = 0
            with torch.no_grad():
                for vb in va_ld:
                    vi = vb['image'].to(device); vl = vb['label'].to(device)
                    with torch.amp.autocast('cuda'):
                        vo = sliding_window_inference(vi, CONFIG['patch_size'], 4, model, overlap=0.5, mode='gaussian')
                        if isinstance(vo, (list,tuple)): vo = vo[0]
                        if vo.dim() == 6: vo = vo[:, 0]
                        vloss = loss_fn(vo, vl)
                    ep_vloss += vloss.item(); vsteps += 1
                    metric((torch.sigmoid(vo) > 0.5).float(), vl)
            v_avg = ep_vloss / max(vsteps, 1)
            dv = metric.aggregate().item()
            log['val_loss'].append(v_avg); log['val_dice'].append(dv)
            elapsed = (time.time() - t0) / 60
            print(f'Ep {epoch:3d}/{CONFIG["epochs"]-1} | Loss={avg:.4f} | ValLoss={v_avg:.4f} | Dice={dv:.4f} | LR={lr:.1e} | {elapsed:.1f}min')
            if dv > best_dice:
                best_dice = dv; patience_ctr = 0
                torch.save({'epoch': epoch, 'best_dice': best_dice,
                    'seg_state_dict': model.state_dict(), 'log': log, 'config': CONFIG}, best_path)
                print('  ✅ New best!')
            else:
                patience_ctr += 1
                if patience_ctr >= CONFIG['patience']:
                    print(f'  Early stop ep {epoch}'); break
        else:
            elapsed = (time.time() - t0) / 60
            print(f'Ep {epoch:3d}/{CONFIG["epochs"]-1} | Loss={avg:.4f} | LR={lr:.1e} | {elapsed:.1f}min')

        torch.save({'epoch': epoch, 'best_dice': best_dice, 'seg_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'log': log, 'patience': patience_ctr}, latest_path)

    elapsed = (time.time() - t0) / 60
    print(f'\n  Fold {fold} done: best Dice = {best_dice:.4f} | {elapsed:.1f} min')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(log['train_loss'], label='Train')
    if log.get('val_loss'):
        vx = list(range(CONFIG['val_interval']-1, len(log['train_loss']), CONFIG['val_interval']))[:len(log['val_loss'])]
        axes[0].plot(vx, log['val_loss'], 'o-', label='Val')
    axes[0].set_title(f'Loss (Fold {fold})'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    if log.get('val_dice'):
        axes[1].plot(log['val_dice'], label='ET Dice', marker='o'); axes[1].legend()
    axes[1].set_title(f'Val Dice (Fold {fold})'); axes[1].set_xlabel('Val Step')
    if log.get('lr'):
        axes[2].plot(log['lr']); axes[2].set_title('LR (WarmRestarts)'); axes[2].set_xlabel('Epoch')
    plt.tight_layout(); plt.savefig(fig_dir / f'metseg_t1c_fold{fold}_curves.png', dpi=150); plt.close()

    return model, best_dice, False

print('Train function ready ✅ (grad_accum=' + str(CONFIG['grad_accum']) + ', CosineWarmRestarts)')

Train function ready ✅ (grad_accum=4, CosineWarmRestarts)


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
#  EMBEDDING EXTRACTION — ROI Crop + Octant Pool + WT Mask-weighted Pool
# ═══════════════════════════════════════════════════════════════════════════

def roi_crop_and_resize(image, label):
    '''Crop to WT bbox + ROI_PADDING, resize to ROI_SIZE.
    image: [1,1,H,W,D],  label: [1,1,H,W,D] (binary WT)
    '''
    wt = label[0, 0]
    nz = wt.nonzero(as_tuple=False)
    if len(nz) == 0:
        return (F.interpolate(image, size=ROI_SIZE, mode='trilinear', align_corners=False),
                F.interpolate(label.float(), size=ROI_SIZE, mode='nearest'))
    lo = torch.clamp(nz.min(0).values - ROI_PADDING, min=0)
    hi = torch.clamp(nz.max(0).values + ROI_PADDING + 1, max=torch.tensor(wt.shape, device=wt.device))
    img_crop = image[:, :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    lbl_crop = label[:, :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    return (F.interpolate(img_crop, size=ROI_SIZE, mode='trilinear', align_corners=False),
            F.interpolate(lbl_crop.float(), size=ROI_SIZE, mode='nearest'))


def octant_pool(feat):
    '''feat: [1,C,H,W,D] → 8×C flat tensor'''
    H, W, D = feat.shape[2:]
    pieces = []
    for hs in [slice(None, H//2), slice(H//2, None)]:
        for ws in [slice(None, W//2), slice(W//2, None)]:
            for ds in [slice(None, D//2), slice(D//2, None)]:
                pieces.append(F.adaptive_avg_pool3d(feat[:,:,hs,ws,ds], 1).flatten())
    return torch.cat(pieces)


def mask_weighted_pool(feat, lbl_roi):
    '''Weight features by binary WT mask.
    feat: [1,C,H,W,D],  lbl_roi: [1,1,H,W,D]
    Returns: C-dim flat tensor
    '''
    H, W, D = feat.shape[2:]
    prob  = F.interpolate(lbl_roi, size=(H,W,D), mode='nearest')
    denom = prob.sum() + 1e-6
    return (feat * prob).sum(dim=[0,2,3,4]) / denom


def extract_embeddings_v2(model, fold):
    '''Full extraction: ROI crop → forward → Octant pool + WT mask pool → save.'''
    model.eval(); model.to(device)
    _feats = {}

    def hook_oct(m, i, o):  _feats['oct']  = (o[0] if isinstance(o,(list,tuple)) else o).detach()
    def hook_neck(m, i, o): _feats['neck'] = (o[0] if isinstance(o,(list,tuple)) else o).detach()

    h_oct  = model.downsamples[-1].register_forward_hook(hook_oct)
    h_neck = model.bottleneck.register_forward_hook(hook_neck) if hasattr(model, 'bottleneck') else None

    all_dicts = get_all_dicts()
    dataset   = CacheDataset(all_dicts, val_transforms, cache_rate=0.3, num_workers=0)
    loader    = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    embeddings = {}; skipped = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc=f'Extracting fold {fold}')):
            image   = batch['image'].to(device)
            label   = batch['label'].to(device)
            patient = batch['patient_dir'][0]
            visit   = batch['visit'][0]
            key     = f'{patient}__{visit}'

            try:
                img_roi, lbl_roi = roi_crop_and_resize(image, label)
                _feats.clear()
                _ = model(img_roi)

                oct_feat  = _feats.get('oct')
                neck_feat = _feats.get('neck', oct_feat)  # fallback if no bottleneck
                if oct_feat is None: skipped += 1; continue

                oct_emb  = octant_pool(oct_feat)                     # 8 × C_oct
                mask_emb = mask_weighted_pool(neck_feat, lbl_roi)    # 1 × C_neck (WT)
                final    = torch.cat([oct_emb, mask_emb]).cpu().numpy()
                embeddings[key] = final

                if i < 3:
                    print(f'  {key}: {final.shape[0]}-dim  (oct={oct_emb.shape[0]} + wt={mask_emb.shape[0]})')

            except Exception as e:
                print(f'  ⚠️ {key}: {e}'); skipped += 1

    h_oct.remove()
    if h_neck: h_neck.remove()
    print(f'  Extracted: {len(embeddings)} | Skipped: {skipped}')

    # Save
    out_path = emb_dir / f'cnn_t1c_embeddings_fold{fold}_v2.npz'
    np.savez(out_path, **embeddings)
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1],
                'embedding_dim': int(final.shape[0])} for k in embeddings}
    with open(emb_dir / f'cnn_t1c_embeddings_fold{fold}_v2_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    dim = list(embeddings.values())[0].shape[0]
    print(f'  ✅ Saved: {out_path.name}  ({len(embeddings)} × {dim}-dim)')
    return embeddings

print('Extraction functions ready ✅')

Extraction functions ready ✅


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
#  3D INFERENCE VISUALIZATION — Ground Truth vs Prediction (Binary WT)
# ═══════════════════════════════════════════════════════════════════════════

from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def render_3d_tumor(ax, mask_3d, color, alpha=0.5, title=''):
    '''Render a binary 3D mask as a surface mesh.'''
    if mask_3d.sum() < 10:
        ax.set_title(f'{title}\n(empty)')
        return
    try:
        verts, faces, _, _ = marching_cubes(mask_3d, level=0.5, step_size=2)
        mesh = Poly3DCollection(verts[faces], alpha=alpha, linewidths=0.1)
        mesh.set_facecolor(color)
        mesh.set_edgecolor([c * 0.7 for c in color[:3]] + [0.3])
        ax.add_collection3d(mesh)
        ax.set_xlim(0, mask_3d.shape[0])
        ax.set_ylim(0, mask_3d.shape[1])
        ax.set_zlim(0, mask_3d.shape[2])
    except Exception as e:
        ax.set_title(f'{title}\n(render failed: {e})')


def visualize_3d_comparison(model, val_loader, fold, n_samples=3):
    '''3D rendering: Ground Truth vs MetSeg T1c Prediction (binary WT).'''
    model.eval()
    wt_color = (0.2, 0.7, 0.3, 0.5)   # green for whole tumor

    samples_done = 0
    with torch.no_grad():
        for idx, batch in enumerate(val_loader):
            if samples_done >= n_samples: break

            image = batch['image'].to(device)
            label = batch['label']

            with torch.amp.autocast('cuda'):
                pred = sliding_window_inference(image, CONFIG['patch_size'], 4, model,
                                                 overlap=0.5, mode='gaussian')
            pred_bin = (torch.sigmoid(pred) > 0.5).float().cpu()

            gt = label[0, 0].numpy()    # (H, W, D) binary WT
            pr = pred_bin[0, 0].numpy()  # (H, W, D) binary WT

            if gt.sum() < 50: continue

            # Dice for this sample
            intersection = (gt * pr).sum()
            union = gt.sum() + pr.sum()
            dice = 2 * intersection / (union + 1e-8)

            fig = plt.figure(figsize=(16, 8))
            fig.suptitle(f'Sample {idx} | WT Dice = {dice:.3f}', fontsize=14, fontweight='bold')

            ax1 = fig.add_subplot(121, projection='3d')
            ax1.set_title('Ground Truth (WT)', fontsize=13, fontweight='bold')
            render_3d_tumor(ax1, gt, wt_color, alpha=0.5)
            ax1.view_init(elev=25, azim=135)
            ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
            ax1.grid(False)
            ax1.xaxis.pane.fill = False; ax1.yaxis.pane.fill = False; ax1.zaxis.pane.fill = False

            ax2 = fig.add_subplot(122, projection='3d')
            ax2.set_title('MetSeg T1c Prediction (WT)', fontsize=13, fontweight='bold')
            render_3d_tumor(ax2, pr, wt_color, alpha=0.5)
            ax2.view_init(elev=25, azim=135)
            ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
            ax2.grid(False)
            ax2.xaxis.pane.fill = False; ax2.yaxis.pane.fill = False; ax2.zaxis.pane.fill = False

            plt.tight_layout()
            save_path = fig_dir / f'metseg_t1c_fold{fold}_3d_sample{idx}.png'
            plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
            plt.show()
            print(f'  Saved: {save_path.name}')
            samples_done += 1

    print(f'\n  ✅ {samples_done} 3D visualizations saved')

print('Visualization function ready ✅')

Visualization function ready ✅


In [11]:
# ╔════════════════════════════════════════════════════════════╗
# ║  MAIN — Train → Extract → Visualize (one fold per run)  ║
# ╚════════════════════════════════════════════════════════════╝

completed = {}
target = None

for f in [0, 1, 2]:
    bp = ckpt_dir / f'metseg_t1c_fold{f}_best.pth'
    lp = ckpt_dir / f'metseg_t1c_fold{f}_latest.pth'
    if bp.exists() and lp.exists():
        c = torch.load(lp, map_location='cpu', weights_only=False)
        if c.get('epoch', -1) >= CONFIG['epochs'] - 1:
            completed[f] = c.get('best_dice', 0)
            print(f'  ✅ Fold {f}: COMPLETE  Dice={completed[f]:.4f}')
            continue
        print(f'  🔄 Fold {f}: PARTIAL (ep {c.get("epoch",0)}/{CONFIG["epochs"]-1})')
    else:
        print(f'  🆕 Fold {f}: NOT STARTED')
    if target is None: target = f

print()

if target is None:
    # ── All folds done — just ensure embeddings + viz are complete ──
    print('🎉 ALL 3 FOLDS COMPLETE!')
    mean = sum(completed.values()) / 3
    print(f'   Mean WT Dice: {mean:.4f}')

    for f in [0, 1, 2]:
        ep = emb_dir / f'cnn_t1c_embeddings_fold{f}_v2.npz'
        if not ep.exists():
            print(f'\n  Extracting embeddings for fold {f}...')
            m = create_segmenter().to(device)
            m.load_state_dict(torch.load(ckpt_dir/f'metseg_t1c_fold{f}_best.pth', map_location=device, weights_only=False)['seg_state_dict'])
            extract_embeddings_v2(m, f)
            del m; torch.cuda.empty_cache()
else:
    # ── TRAIN one fold ──
    print(f'▶ Training FOLD {target}')
    print('=' * 60)
    model, best, was_skipped = train_fold(target)
    completed[target] = best

    # ── EXTRACT embeddings immediately ──
    print(f'\n  Extracting embeddings for fold {target}...')
    torch.cuda.empty_cache()

    # Reload from best checkpoint for clean extraction
    best_ckpt = torch.load(ckpt_dir / f'metseg_t1c_fold{target}_best.pth', map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['seg_state_dict'])
    extract_embeddings_v2(model, target)

    # ── VISUALIZE 3D ──
    print('\n' + '=' * 60)
    print('  3D INFERENCE VISUALIZATION')
    print('=' * 60)
    _, vis_val_dicts = get_fold_dicts(target)
    vis_ds = CacheDataset(vis_val_dicts, val_transforms, cache_rate=1.0, num_workers=0)
    vis_ld = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    visualize_3d_comparison(model, vis_ld, target, n_samples=3)

    del model; torch.cuda.empty_cache()

    remaining = [f for f in [0,1,2] if f not in completed]
    print(f'\n✅ Fold {target} done — Dice={best:.4f}')
    if remaining: print(f'→ Relaunch to train fold {remaining[0]}')
    else:         print(f'🎉 ALL FOLDS COMPLETE! Mean={sum(completed.values())/3:.4f}')

# ── Summary ────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print(f'  Files saved to {OUTPUT_ROOT}:')
for p in sorted(OUTPUT_ROOT.rglob('*')):
    if p.is_file():
        print(f'    {p.relative_to(OUTPUT_ROOT)} ({p.stat().st_size/1024/1024:.1f} MB)')
total_mb = sum(p.stat().st_size for p in OUTPUT_ROOT.rglob('*') if p.is_file()) / (1024*1024)
print(f'\n  Total output: {total_mb:.1f} MB / 19,000 MB limit')
print('=' * 60)

  🆕 Fold 0: NOT STARTED
  🆕 Fold 1: NOT STARTED
  🆕 Fold 2: NOT STARTED

▶ Training FOLD 0
  Train: 118 | Val: 52


Loading dataset: 100%|██████████| 52/52 [00:20<00:00,  2.59it/s]


  Loaded: 168/168 params
  Adapted 18 layers:
    input_block.conv1.conv.weight: (32, 4, 3, 3, 3)→(32, 1, 3, 3, 3) (T1c ch=1)
    input_block.conv3.conv.weight: (32, 4, 1, 1, 1)→(32, 1, 1, 1, 1) (T1c ch=1)
    output_block.conv.conv.weight: (3, 32, 1, 1, 1)→(1, 32, 1, 1, 1) (ET ch=2)
    output_block.conv.conv.bias: (3,)→(1,) (ET ch=2)
    deep_supervision_heads.0.conv.conv.weight: (3, 64, 1, 1, 1)→(1, 64, 1, 1, 1) (ET ch=2)
  ✅ Pretrained weights loaded (T1c input, ET output)
Ep   0/99 | Loss=1.1156 | ValLoss=0.4960 | Dice=0.0261 | LR=1.0e-05 | 1.9min
  ✅ New best!
Ep   1/99 | Loss=1.0184 | ValLoss=0.4649 | Dice=0.0952 | LR=4.0e-05 | 3.9min
  ✅ New best!
Ep   2/99 | Loss=0.8897 | ValLoss=0.4292 | Dice=0.1623 | LR=7.0e-05 | 6.0min
  ✅ New best!
Ep   3/99 | Loss=0.8374 | ValLoss=0.3942 | Dice=0.2296 | LR=1.0e-04 | 8.1min
  ✅ New best!
Ep   4/99 | Loss=0.8133 | ValLoss=0.3580 | Dice=0.3038 | LR=1.0e-04 | 10.2min
  ✅ New best!
Ep   5/99 | Loss=0.7813 | ValLoss=0.3497 | Dice=0.3221 | LR=1.

Extracting fold 0:   2%|▏         | 3/170 [00:00<00:22,  7.51it/s]

  P02__baseline: 2368-dim  (oct=2048 + wt=320)
  P02__fu1: 2368-dim  (oct=2048 + wt=320)
  P02__fu2: 2368-dim  (oct=2048 + wt=320)


Extracting fold 0: 100%|██████████| 170/170 [00:48<00:00,  3.54it/s]


  Extracted: 170 | Skipped: 0
  ✅ Saved: cnn_t1c_embeddings_fold0_v2.npz  (170 × 2368-dim)

  3D INFERENCE VISUALIZATION


Loading dataset: 100%|██████████| 52/52 [00:17<00:00,  2.97it/s]


  Saved: metseg_t1c_fold0_3d_sample0.png
  Saved: metseg_t1c_fold0_3d_sample1.png
  Saved: metseg_t1c_fold0_3d_sample2.png

  ✅ 3 3D visualizations saved

✅ Fold 0 done — Dice=0.4774
→ Relaunch to train fold 1

  Files saved to /kaggle/working/phase2_metseg_t1c_outputs:
    checkpoints/metseg_t1c_fold0_best.pth (63.7 MB)
    checkpoints/metseg_t1c_fold0_latest.pth (190.9 MB)
    embeddings/cnn_t1c_embeddings_fold0_v2.npz (1.6 MB)
    embeddings/cnn_t1c_embeddings_fold0_v2_meta.json (0.0 MB)
    figures/metseg_t1c_fold0_3d_sample0.png (0.2 MB)
    figures/metseg_t1c_fold0_3d_sample1.png (0.1 MB)
    figures/metseg_t1c_fold0_3d_sample2.png (0.1 MB)
    figures/metseg_t1c_fold0_curves.png (0.1 MB)

  Total output: 256.8 MB / 19,000 MB limit


In [12]:
# ── Quick Diversity + Verification ────────────────────────────────────────
import random

print('── Embedding Verification ──')
for fold in [0, 1, 2]:
    p = emb_dir / f'cnn_t1c_embeddings_fold{fold}_v2.npz'
    if not p.exists(): print(f'  ❌ fold{fold}: NOT FOUND'); continue
    data  = np.load(p)
    keys  = list(data.keys())
    vals  = np.stack([data[k] for k in keys])
    norms = np.linalg.norm(vals, axis=1)
    vn    = vals / (norms[:,None] + 1e-8)
    n_p   = min(50, len(keys)*(len(keys)-1)//2)
    pairs = random.sample([(i,j) for i in range(len(keys)) for j in range(i+1,len(keys))], n_p)
    sims  = [float(np.dot(vn[i], vn[j])) for i,j in pairs]
    div   = 100 * np.mean(np.array(sims) < 0.95)
    st    = '✅' if div > 20 else '⚠️ low'
    print(f'  fold{fold}: {len(keys)} × {vals.shape[1]}-dim | '
          f'norm [{norms.min():.2f},{norms.max():.2f}] | '
          f'cos={np.mean(sims):.3f} diverse={div:.0f}% {st}')

── Embedding Verification ──
  fold0: 170 × 2368-dim | norm [0.86,3.59] | cos=0.570 diverse=100% ✅
  ❌ fold1: NOT FOUND
  ❌ fold2: NOT FOUND


In [13]:
# ════════════════════════════════════════════════════════════
# TEST M7: Longitudinal Embedding-Volume Correlation
# Core validity check: If delta_emb correlates with delta_vol
# → embeddings genuinely track ET progression
# ════════════════════════════════════════════════════════════
import numpy as np, nibabel as nib
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt

def get_et_volume_mm3(mask_path):
    try:
        img = nib.load(str(mask_path))
        mask = img.get_fdata()
        zooms = img.header.get_zooms()
        vox_vol = float(zooms[0]) * float(zooms[1]) * float(zooms[2])
        return int(np.sum(mask == 2)) * vox_vol  # label 2 = ET
    except Exception:
        return None

def run_m7_test(emb_path, data_root, model_name, fold_id):
    npz = np.load(str(emb_path), allow_pickle=True)
    keys = list(npz.files)
    print("  Embeddings loaded:", len(keys))

    patients = sorted(set(k.split("__")[0] for k in keys))
    timepoints = ["baseline", "fu1", "fu2"]

    emb_dists, vol_changes, labels = [], [], []

    for pid in patients:
        avail = [tp for tp in timepoints if pid + "__" + tp in keys]
        if len(avail) < 2:
            continue
        for i in range(len(avail) - 1):
            tp0, tp1 = avail[i], avail[i+1]
            e0 = npz[pid + "__" + tp0].astype(np.float64)
            e1 = npz[pid + "__" + tp1].astype(np.float64)
            d_emb = float(np.linalg.norm(e1 - e0))

            root = Path(data_root)
            m0 = next(root.rglob("*" + pid + "*" + tp0 + "*seg*"), None) or \
                 next(root.rglob("*" + pid + "*" + tp0 + "*.nii*"), None)
            m1 = next(root.rglob("*" + pid + "*" + tp1 + "*seg*"), None) or \
                 next(root.rglob("*" + pid + "*" + tp1 + "*.nii*"), None)

            v0 = get_et_volume_mm3(m0) if m0 else None
            v1 = get_et_volume_mm3(m1) if m1 else None

            if v0 is not None and v1 is not None:
                d_vol = v1 - v0
                emb_dists.append(d_emb)
                vol_changes.append(d_vol)
                labels.append(pid + " " + tp0 + "->" + tp1)
                print("  " + pid + " (" + tp0 + "->" + tp1 + "): delta_emb=" + str(round(d_emb,3)) + " | delta_ETvol=" + str(round(d_vol,1)) + "mm3")

    if len(emb_dists) < 3:
        print("  WARNING: only " + str(len(emb_dists)) + " pairs with volume - skipping stats")
        return None

    e = np.array(emb_dists)
    v = np.array(vol_changes)
    abs_v = np.abs(v)

    pr, pp = stats.pearsonr(e, abs_v)
    sr, sp = stats.spearmanr(e, abs_v)

    print("\n  ── M7 Results (n=" + str(len(e)) + " pairs) ──")
    print("  Pearson  r=" + str(round(pr,3)) + "  p=" + str(round(pp,4)))
    print("  Spearman r=" + str(round(sr,3)) + "  p=" + str(round(sp,4)))
    if pp < 0.05:
        print("  SIGNIFICANT: delta_embedding tracks |delta_ET_volume| (p<0.05)")
    elif pp < 0.10:
        print("  TREND: delta_embedding tracks |delta_ET_volume| (p<0.10)")
    else:
        print("  NO CORRELATION: embeddings do not track volume change")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(model_name + " — Test M7: Longitudinal Embedding-Volume Correlation (fold " + str(fold_id) + ")",
                 fontsize=12, fontweight="bold")

    ax1.scatter(v, e, c=np.arange(len(v)), cmap="viridis", s=80, alpha=0.8)
    for i, lbl in enumerate(labels):
        ax1.annotate(lbl.split(" ")[0], (v[i], e[i]), fontsize=7, alpha=0.7, ha="center", va="bottom")
    xf = np.linspace(v.min(), v.max(), 100)
    ax1.plot(xf, np.poly1d(np.polyfit(v, e, 1))(xf), "r--", alpha=0.7)
    ax1.axvline(0, color="gray", lw=0.8, ls=":")
    ax1.set_xlabel("Delta ET Volume (mm3)  [negative=regression]")
    ax1.set_ylabel("Delta Embedding (L2 distance)")
    ax1.set_title("Signed | Pearson r=" + str(round(pr,3)) + " p=" + str(round(pp,3)))
    ax1.grid(alpha=0.3)

    ax2.scatter(abs_v, e, c=np.arange(len(abs_v)), cmap="plasma", s=80, alpha=0.8)
    xf2 = np.linspace(abs_v.min(), abs_v.max(), 100)
    ax2.plot(xf2, np.poly1d(np.polyfit(abs_v, e, 1))(xf2), "r--", alpha=0.7)
    ax2.set_xlabel("|Delta ET Volume| (mm3)")
    ax2.set_ylabel("Delta Embedding (L2 distance)")
    ax2.set_title("Absolute | Spearman rho=" + str(round(sr,3)) + " p=" + str(round(sp,3)))
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    out = fig_dir / (model_name.lower().replace(" ","_") + "_fold" + str(fold_id) + "_M7_longi_corr.png")
    plt.savefig(str(out), dpi=150, bbox_inches="tight"); plt.close()
    print("  Saved: " + out.name)

    return {"pearson_r": pr, "pearson_p": pp, "spearman_r": sr, "spearman_p": sp, "n_pairs": len(e)}

print("=" * 60)
print("  TEST M7 — Longitudinal Embedding-Volume Correlation")
print("=" * 60)
for fold_id in range(3):
    candidates = list(Path("/kaggle/working").rglob("*embeddings*fold" + str(fold_id) + "*v2.npz"))
    if not candidates:
        print("  fold " + str(fold_id) + ": embeddings not found - skip")
        continue
    result = run_m7_test(candidates[0], DATA_ROOT, MODEL_NAME, fold_id)
    if result:
        print("  fold " + str(fold_id) + " completed: r=" + str(round(result["pearson_r"],3)))


  TEST M7 — Longitudinal Embedding-Volume Correlation
  Embeddings loaded: 170
  fold 1: embeddings not found - skip
  fold 2: embeddings not found - skip
